# 05 Join Optimization

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Make slow joins fast with a repeatable checklist: shrink the inputs (filter, select, pre-aggregate), pick the right strategy (broadcast vs sort-merge), make sure keys are unique, clean and type-matched, use semi and anti joins for existence checks, and let the optimizer prune with dynamic file pruning.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"A join is slow. How do you debug it?" is one of the scenarios this learning path asks you to rehearse, and joins are where most Spark time is spent in real pipelines. A few habits fix the majority of slow joins without any cluster tuning.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The join checklist</b>
<ol><li><b>Shrink both sides first</b>: filter rows, select only needed columns, and aggregate the big side before joining if the result only needs totals</li><li><b>Pick the strategy</b>: broadcast when one side is small (or becomes small after filtering). Otherwise sort-merge, with AQE</li><li><b>Check the keys</b>: unique on the dimension side? Same data type on both sides? Nulls or default values (<code>'unknown'</code>, <code>-1</code>) piling up on one key?</li><li><b>Use the right join type</b>: <code>left_semi</code> / <code>left_anti</code> for "exists / not exists" instead of join + distinct</li><li><b>Help pruning</b>: filter the dimension so dynamic file/partition pruning can skip fact data. Cluster big tables by join keys</li><li><b>Measure</b> in the query profile: rows in and out, shuffle bytes, task skew</li></ol>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A daily report joined 3 billion clickstream rows to 20 million user profiles, then grouped by country. It took 70 minutes: both sides were shuffled with all 80 columns. Aggregating clicks per user first (3 billion → 20 million rows), selecting 3 columns from profiles, and joining the two 20-million-row results brought it to 6 minutes. The join had been doing 150 times more work than needed.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a large <code>clicks</code> fact table and a <code>users</code> dimension with a few realistic problems: an extra wide column, a key stored as a string on one side, and many clicks with an unknown user (<code>-1</code>).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Row counts and schemas of both tables.
</div>

In [0]:
import time
from pyspark.sql import functions as F

clicks = spark.range(0, 3_000_000, 1, numPartitions=8).select(
    F.col("id").alias("click_id"),
    F.when(F.col("id") % 10 == 0, F.lit(-1)).otherwise(F.col("id") % 200_000).alias("user_id"),     # 10% unknown users
    F.expr("date_add(date'2024-11-01', pmod(hash(id), 30))").alias("click_date"),
    F.sha2(F.col("id").cast("string"), 256).alias("url_hash"),
)
users = spark.range(200_000).select(
    F.col("id").cast("string").alias("user_id"),                                                     # key stored as STRING here
    F.element_at(F.array(*[F.lit(c) for c in ["IN", "UK", "AE", "US"]]), (F.col("id") % 4 + 1).cast("int")).alias("country"),
    F.sha2(F.col("id").cast("string"), 512).alias("profile_blob"),                                  # wide column nobody needs
)
print("clicks:", clicks.count(), "| users:", users.count())
users.printSchema()

def timed(label, df):
    start = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:<55} {time.time() - start:6.2f}s")

## 1. Baseline: the naive join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins everything, then groups by country.<br><br>
<b>Why it matters</b><br>This is the version people write first. It shuffles every column of both sides, compares keys of different types, and carries the 10% unknown-user rows all the way.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A baseline timing, and clicks per country (<code>-1</code> clicks drop out, because no user has that ID).
</div>

In [0]:
naive = clicks.join(users, clicks.user_id == users.user_id).groupBy("country").count()
timed("naive join", naive)
naive.orderBy("country").show()

## 2. Fix the key type mismatch

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Looks at the plan of the naive join, then casts the dimension key to the same type as the fact key.<br><br>
<b>Why it matters</b><br>Joining <code>BIGINT</code> to <code>STRING</code> forces an implicit cast on <b>every row</b> of one side, can block pushdown and data skipping, and with ANSI mode, bad strings fail at runtime. Align key types at the source (Silver), not in every query.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The naive plan contains a <code>cast(... as bigint)</code> (or string) on the join key. The aligned version has no cast in the join condition.
</div>

In [0]:
import io, contextlib

def plan(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

print("cast in naive join condition:", "cast(" in [l for l in plan(naive).splitlines() if "Join" in l][0])
users_typed = users.withColumn("user_id", F.col("user_id").cast("bigint"))
aligned = clicks.join(users_typed, "user_id").groupBy("country").count()
print("cast in aligned join condition:", "cast(" in [l for l in plan(aligned).splitlines() if "Join" in l][0])

## 3. Shrink the inputs: filter, select, pre-aggregate

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Applies the checklist step by step: drop unknown users, select only needed columns, and aggregate clicks per user <b>before</b> the join.<br><br>
<b>Why it matters</b><br>After pre-aggregation, the join processes 200,000 rows instead of 3 million, and the shuffle carries a few small columns instead of 64-character hashes. The result is identical.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Steps 1 and 2 are clearly faster than the naive join, and the final result equals the naive result. Step 3 may <b>not</b> be faster at this demo size: it adds an aggregation, and 3 million rows only shrink to 180,000. Pre-aggregation pays off when each key has many rows (billions of facts, millions of keys), which is why you measure each step instead of assuming.
</div>

In [0]:
step1 = (clicks.filter(F.col("user_id") != -1)
         .join(users_typed, "user_id").groupBy("country").count())
step2 = (clicks.filter(F.col("user_id") != -1).select("user_id")
         .join(users_typed.select("user_id", "country"), "user_id").groupBy("country").count())
step3 = (clicks.filter(F.col("user_id") != -1).groupBy("user_id").agg(F.count("*").alias("clicks"))
         .join(users_typed.select("user_id", "country"), "user_id")
         .groupBy("country").agg(F.sum("clicks").alias("count")))

timed("1. filter unknown users", step1)
timed("2. + select only needed columns", step2)
timed("3. + pre-aggregate clicks per user", step3)
print("same result:", sorted(naive.collect()) == sorted((r["country"], r["count"]) for r in step3.collect()))

## 4. Broadcast when one side is small

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins with an explicit broadcast of the (now narrow) users dimension.<br><br>
<b>Why it matters</b><br>With only <code>user_id</code> and <code>country</code>, the users side is small enough to broadcast, so the clicks side isn't shuffled at all. Selecting columns first is what makes a dimension broadcastable.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>BroadcastHashJoin</code> in the plan and the fastest timing so far.
</div>

In [0]:
bcast = (clicks.filter(F.col("user_id") != -1).select("user_id")
         .join(F.broadcast(users_typed.select("user_id", "country")), "user_id")
         .groupBy("country").count())
print("strategy:", "BroadcastHashJoin" if "BroadcastHashJoin" in plan(bcast) else "other")
timed("4. broadcast the narrow dimension", bcast)

## 5. Existence checks: semi and anti joins

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Finds users who clicked at least once, first with join + distinct, then with a <code>left_semi</code> join, and users who never clicked with a <code>left_anti</code> join.<br><br>
<b>Why it matters</b><br>A normal join duplicates each user once per click, and then <code>distinct</code> has to remove the duplicates with another shuffle. A semi join stops at the first match and returns each user once.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Both methods return the same number of active users, and 20,000 users never clicked. At this size the timings are similar. The semi join's advantage grows with the number of matches per key, because it never builds the duplicated rows that <code>distinct</code> has to remove.
</div>

In [0]:
via_join = users_typed.join(clicks.select("user_id"), "user_id").select("user_id", "country").distinct()
via_semi = users_typed.join(clicks.select("user_id"), "user_id", "left_semi").select("user_id", "country")
timed("join + distinct", via_join)
timed("left_semi join", via_semi)
print("same count:", via_join.count() == via_semi.count())
print("users who never clicked:", users_typed.join(clicks.select("user_id"), "user_id", "left_anti").count())

## 6. Detect row explosion before it happens

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Checks a dimension for duplicate keys before joining, with a reusable function.<br><br>
<b>Why it matters</b><br>Duplicate keys multiply rows silently (lesson 01_17). A two-line check before important joins prevents wrong totals and wasted compute.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>users_typed</code> has no duplicate keys. A deliberately broken copy reports duplicates.
</div>

In [0]:
def assert_unique(df, key, name):
    dupes = df.groupBy(key).count().filter("count > 1").limit(5).collect()
    print(f"{name}: {'OK, keys unique' if not dupes else f'DUPLICATE KEYS e.g. {[r[key] for r in dupes]}'}")
    return not dupes

assert_unique(users_typed, "user_id", "users_typed")
assert_unique(users_typed.unionByName(users_typed.limit(3)), "user_id", "broken copy")

## 7. Dynamic file pruning: filter the dimension, skip the facts

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the facts as a table clustered by date, joins it to a filtered calendar dimension, and checks the plan for dynamic pruning.<br><br>
<b>Why it matters</b><br>When the filter is on the <b>dimension</b> (for example "weekends in November"), Databricks can use the dimension's matching keys at runtime to skip fact files or partitions that can't match. That needs a selective filter on the dimension and a fact table laid out by the join key.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
On Databricks: <code>dynamic pruning in plan: True</code> (the fact scan has a <code>dynamicpruning</code> filter, and the query profile shows files or partitions pruned). In open-source Spark with Delta this can print <code>False</code>, because dynamic file pruning is a Databricks optimization.
</div>

In [0]:
DB = "workspace.optimization"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
facts_t = f"{DB}.clicks_by_date"
clicks.select("click_id", "user_id", "click_date").write.mode("overwrite").partitionBy("click_date").saveAsTable(facts_t)
calendar = spark.sql("SELECT explode(sequence(date'2024-11-01', date'2024-11-30', interval 1 day)) AS click_date") \
    .withColumn("is_weekend", F.dayofweek("click_date").isin(1, 7))
calendar.write.mode("overwrite").saveAsTable(f"{DB}.calendar")

weekend = spark.table(facts_t).join(spark.table(f"{DB}.calendar").filter("is_weekend"), "click_date").groupBy("click_date").count()
print("dynamic pruning in plan:", "dynamicpruning" in plan(weekend).lower())
for t in [facts_t, f"{DB}.calendar"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

## Under the hood

```
naive:      clicks (3M rows, 4 cols) ──shuffle──┐
            users  (200k rows, wide) ──shuffle──┴─ sort-merge join ─ group by country
optimized:  clicks ─ filter ─ select ─ partial agg ─ (200k rows) ─┐
            users  ─ select 2 cols ─ BROADCAST ───────────────────┴─ hash join ─ group by
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> a sort-merge join shuffles both sides. Every byte you remove before it saves disk, network and memory.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> a broadcast hash join leaves the big side in place.

In [0]:
step3.explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: join on columns with different types</b><br>
Implicit casts on every row and broken pushdown. Align types in Silver.<br><br>
**⛔ Problem: huge task for the key <code>-1</code>, <code>0</code>, <code>''</code> or <code>NULL</code>**<br>Placeholder keys create skew and never match. Filter them out before the join, or handle them separately.<br><br>
**⛔ Problem: join + <code>distinct</code> to check existence**<br>Use <code>left_semi</code>, and <code>left_anti</code> for "not exists".<br><br>
<b>⛔ Problem: carrying wide columns through the join</b><br>Select only what the result needs, before joining.<br><br>
<b>⛔ Problem: totals inflated after a join</b><br>Duplicate dimension keys. Assert uniqueness before joining.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. A join is slow. How do you debug it?</b><br>
I open the query profile and check the join strategy, the rows in and out (explosion), the shuffle bytes on each side, and the task time distribution (skew). Then I apply the checklist: filter and select before joining, pre-aggregate the big side if only totals are needed, broadcast a small side, make sure keys are unique, typed consistently and free of placeholder values, and handle skewed keys. Then I measure again.<br><br>

<b>🎤 2. Why pre-aggregate before a join?</b><br>
If the result only needs totals per key, aggregating the big side first reduces it to one row per key, so the join shuffles and compares far less data, with the same result.<br><br>

<b>🎤 3. When should you use a semi or anti join?</b><br>
To check whether matching rows exist (semi) or don't exist (anti). They return each left row at most once and only the left columns, avoiding duplicates and an extra distinct.<br><br>

<b>🎤 4. What happens when join keys have different data types?</b><br>
Spark adds a cast on one side for every row, which costs CPU, can prevent pushdown and data skipping, and can fail at runtime with ANSI mode if values don't convert.<br><br>

<b>🎤 5. What is dynamic file pruning?</b><br>
When a fact table is joined to a filtered dimension, Databricks uses the dimension's matching key values at runtime to skip fact files (or partitions) that can't match, which works best when the fact table is clustered or partitioned by the join key.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A query joins a 2-billion-row fact table to a dimension and then sums a measure per dimension attribute. Which change typically reduces the join cost most?</b><br>
A. Adding more columns to the join output<br>
B. Aggregating the fact table by the join key before the join<br>
C. Converting the dimension to CSV<br>
D. Using a cross join<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which join type returns customers that have at least one order, without duplicating customers?</b><br>
A. <code>inner</code><br>
B. <code>left_semi</code><br>
C. <code>full_outer</code><br>
D. <code>cross</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Count clicks per country for November weekends only, applying every item of the checklist, and compare with a naive version</li><li>Find the users who clicked on more than 10 distinct days, joined to their country, as efficiently as you can</li><li>Add <code>assert_unique</code> to your pipeline before each dimension join</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Filter, select, pre-aggregate, broadcast
weekend_days = [r[0] for r in spark.sql("SELECT explode(sequence(date'2024-11-01', date'2024-11-30', interval 1 day)) AS d")
                .filter(F.dayofweek("d").isin(1, 7)).collect()]
optimized = (clicks.filter((F.col("user_id") != -1) & F.col("click_date").isin(weekend_days))
             .groupBy("user_id").agg(F.count("*").alias("clicks"))
             .join(F.broadcast(users_typed.select("user_id", "country")), "user_id")
             .groupBy("country").agg(F.sum("clicks").alias("clicks")))
naive_weekend = clicks.join(users, clicks.user_id == users.user_id).filter(F.dayofweek("click_date").isin(1, 7)).groupBy("country").count()
# At demo size the timings are close (or reversed); the optimized plan shuffles far less data, which dominates at scale
timed("naive weekend clicks", naive_weekend)
timed("optimized weekend clicks", optimized)

# 2. Users with > 10 distinct click days: aggregate first, then a small broadcast join
assert_unique(users_typed, "user_id", "users")
(clicks.filter(F.col("user_id") != -1).groupBy("user_id").agg(F.countDistinct("click_date").alias("days"))
 .filter("days > 10").join(F.broadcast(users_typed.select("user_id", "country")), "user_id")
 .groupBy("country").count().orderBy("country").show())

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Shrink first: filter, select, pre-aggregate the big side</li><li>Broadcast small (or small-after-select) sides. Sort-merge + AQE for large-large joins</li><li>Keys: unique on the dimension, same type on both sides, no placeholder or null hot keys</li><li><code>left_semi</code> / <code>left_anti</code> for existence checks</li><li>Cluster or partition facts by join keys so dynamic pruning can skip data</li><li>Measure every change in the query profile</li></ul>
</div>

| Problem | Fix |
|---|---|
| Too much data in the join | `filter`, `select`, `groupBy` before joining |
| Small dimension shuffled | `F.broadcast(dim.select(...))` |
| Type mismatch | Cast once in Silver |
| Placeholder keys | Filter or handle separately |
| Exists / not exists | `left_semi`, `left_anti` |
| Row explosion | `assert_unique(dim, key)` |

## Git commit

```
git add 05_optimization/05_join_optimization.ipynb
git commit -m "add 05_05 join optimization notebook"
```